## SRP237993

**paper:** [PMID: 32770934](https://pmc.ncbi.nlm.nih.gov/articles/PMC7414754/) - Distinct evolutionary trajectories of V1R clades across mouse species, 2020

**date, curator:** 2026-04-21, Sara Carsanaro

**notes**
* rejected pacbio library
* updated info organ to Vomeronasal organ - epithelium based on methods stating they took the epithelium only

### annotation summary

In [32]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Vomeronasal organ - epithelium,UBERON:0003367,epithelium of vomeronasal organ,perfect match


In [33]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP237993"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 23/23 [00:30<00:00,  1.31s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [13]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,,,,,,Vomeronasal organ - epithelium,Adult,,,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,,,,,,Vomeronasal organ - epithelium,Adult,,,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,,,,,,Vomeronasal organ - epithelium,Adult,,,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,6_SFM_VNO_F.101_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
3,SRX7400226,SRP237993,NextSeq 500,SRS5847892,,,,,,Vomeronasal organ - epithelium,Adult,,,,pool

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [14]:
unique_sorted(library, "infoOrgan")

['Vomeronasal organ - epithelium']


In [15]:

# all
library.loc[:,'anatId'] = 'UBERON:0003367'
library.loc[:,'anatName'] = 'epithelium of vomeronasal organ'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,,,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,,,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,,,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Re

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [16]:
unique_sorted(library, "infoStage")

['Adult']


In [17]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [18]:
#library.loc[:,'sex'] = ''
library.loc[library["sex"] == "pooled male and female", "sex"] = "mixed"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module","full_length,nan","nan,polyA",,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [19]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,,21/04/2026,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E76

#### globin, replicates

In [20]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

    #libraryId       SRSId
20  SRX7400208  SRS5847876
21  SRX7400207  SRS5847876
6   SRX7400223  SRS5847887
9   SRX7400219  SRS5847887


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_36104/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [21]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-21'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.103,SAMN13618368,,,,,,,,,,SAC,2026-04-21,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,8_SFM_VNO_F.103_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.102,SAMN13618367,,,,,,,,,,SAC,2026-04-21,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1, NEB #E7600). VNO libraries were sequenced as 125 bp paired-end reads on Illumina HiSeq2500 platform through the Biotechnology Resource Center (Institute of Biotechnology) at Cornell University",,7_SFM_VNO_F.102_RNASeq,,,,Adult,,TRANSCRIPTOMIC,PolyA
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.101,SAMN13618366,,,,,,,,,,SAC,2026-04-21,"VNO epithelia were dissected from at least one male and one female from each species and subsequently pooled to obtain V1R repertoires unbiased to a particular sex. Total RNA was extracted from VNO tissues using the Qiagen RNeasy kit, and subsequently quantified using QuBit Fluorometric Quantitation. RNA sequencing libraries were generated using the NEBNext Ultra RNA Library Prep Kit for Illumina (NEB #E7530). NEBNext Poly(A) mRNA Magnetic Isolation Module (NEB #E7490) was used for RNA Isolation. Sequences were indexed using the NEBNext Multiplex Oligos for Illumina (Dual Index Primers Set 1,

#### comments

In [22]:
library.loc[:,'comment'] = 'PMID: 32770934'

#### save complete file with correct columns

In [23]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.103,SAMN13618368,,,,,,,PMID: 32770934,,,SAC,2026-04-21
1,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.102,SAMN13618367,,,,,,,PMID: 32770934,,,SAC,2026-04-21
2,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.101,SAMN13618366,,,,,,,PMID: 32770934,,,SAC,2026-04-21
3,SRX7400226,SRP237993,NextSeq 500,SRS5847892,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,PAH,,10093,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,PAH.VNO.Pool,SAMN13618365,,,,,,,PMID: 32770934,,,SAC,2026-04-21
4,SRX7400225,SRP237993,NextSeq 500,SRS5847890,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,CAR,,10089,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,CAR.VNO.Pool.2,SAMN13618364,,,,,,,PMID: 32770934,,,SAC,2026-04-21
5,SRX7400224,SRP237993,Illumina MiSeq,SRS5847891,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,CAR,,10089,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,CAR.VNO.Pool.1,SAMN13618363,,,,,,,PMID: 32770934,,,SAC,2026-04-21
6,SRX7400223,SRP237993,NextSeq 500,SRS5847887,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,XBS,,10100,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,2,XBS.VNO.Pool,SAMN13618362,,,,,,,PMID: 32770934,,,SAC,2026-04-21
7,SRX7400221,SRP237993,Illumina HiSeq 2500,SRS5847889,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.136,SAMN13618380,,,,,,,PMID: 32770934,,,SAC,2026-04-21
8,SRX7400220,SRP237993,Illumina HiSeq 2500,SRS5847897,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,polyA,,,SFM.VNO.F.135,SAMN13618379,,,,,,,PMID: 32770934,,,SAC,2026-04-21
9,SRX7400219,SRP237993,Illumi

### experiment annotations

In [24]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP237993,Vomeronasal organ transcriptomes for 5 mouse species within the Mus genus,"Among rodents, olfaction is crucial for a wide range of social behaviors. The vomeronasal olfactory system in particular plays an important role in mediating social communication, including the detection of pheromones and recognition signals. Currently, very few vomeronasal receptors have known ligands, which severely limits our understanding of chemosensory-driven social communication. This dataset was generated to create high-quality vomeronasal receptor (VR) repertoires among 5 mouse species closely related to the house mouse. Exploring the evolution of these receptors can provide insight into the functional roles of receptor subtypes as well as the dynamics of gene family evolution.",SRA,,,,"NEBNext Ultra RNA Library Prep Kit, Poly(A) mRNA Magnetic Isolation Module","full_length, nan",,PRJNA596328,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [25]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

22

In [26]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP237993,Vomeronasal organ transcriptomes for 5 mouse species within the Mus genus,"Among rodents, olfaction is crucial for a wide range of social behaviors. The vomeronasal olfactory system in particular plays an important role in mediating social communication, including the detection of pheromones and recognition signals. Currently, very few vomeronasal receptors have known ligands, which severely limits our understanding of chemosensory-driven social communication. This dataset was generated to create high-quality vomeronasal receptor (VR) repertoires among 5 mouse species closely related to the house mouse. Exploring the evolution of these receptors can provide insight into the functional roles of receptor subtypes as well as the dynamics of gene family evolution.",SRA,partial,Bgee 1K,22,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,,PRJNA596328,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [27]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '32770934'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7414754/'
experiment.loc[:,'DOI'] = '10.1186/s12862-020-01662-z'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP237993,Vomeronasal organ transcriptomes for 5 mouse species within the Mus genus,"Among rodents, olfaction is crucial for a wide range of social behaviors. The vomeronasal olfactory system in particular plays an important role in mediating social communication, including the detection of pheromones and recognition signals. Currently, very few vomeronasal receptors have known ligands, which severely limits our understanding of chemosensory-driven social communication. This dataset was generated to create high-quality vomeronasal receptor (VR) repertoires among 5 mouse species closely related to the house mouse. Exploring the evolution of these receptors can provide insight into the functional roles of receptor subtypes as well as the dynamics of gene family evolution.",SRA,partial,Bgee 1K,22,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,,PRJNA596328,32770934,https://pmc.ncbi.nlm.nih.gov/articles/PMC7414754/,10.1186/s12862-020-01662-z,,


#### comments

In [28]:
experiment.loc[:,'comment'] = 'rejected pacbio library'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP237993,Vomeronasal organ transcriptomes for 5 mouse species within the Mus genus,"Among rodents, olfaction is crucial for a wide range of social behaviors. The vomeronasal olfactory system in particular plays an important role in mediating social communication, including the detection of pheromones and recognition signals. Currently, very few vomeronasal receptors have known ligands, which severely limits our understanding of chemosensory-driven social communication. This dataset was generated to create high-quality vomeronasal receptor (VR) repertoires among 5 mouse species closely related to the house mouse. Exploring the evolution of these receptors can provide insight into the functional roles of receptor subtypes as well as the dynamics of gene family evolution.",SRA,partial,Bgee 1K,22,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRNA Magnetic Isolation Module",full_length,,PRJNA596328,32770934,https://pmc.ncbi.nlm.nih.gov/articles/PMC7414754/,10.1186/s12862-020-01662-z,,rejected pacbio library


#### save complete file

In [29]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [30]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [31]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [34]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [35]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62197,SRX736539,SRP049070,Illumina HiSeq 2000,SRS724635,UBERON:0000468,multicellular organism,UBERON:0034919,juvenile stage,,whole body,juvenile,other,full sampling,other,H,Guadeloupe,homozygous for R haplotype of Guadeloupe Resis...,6526,TruSeq RNA Library Prep Kit,full_length,polyA,,1,GRC_R,SAMN03112928,,,,,,,"PMID:25775214, We extracted RNA from whole bod...",not exposed to parasite,,ANN,2026-04-21
62198,SRX736535,SRP049070,Illumina HiSeq 2000,SRS724635,UBERON:0000468,multicellular organism,UBERON:0034919,juvenile stage,,whole body,juvenile,other,full sampling,other,H,Guadeloupe,homozygous for R haplotype of Guadeloupe Resis...,6526,TruSeq RNA Library Prep Kit,full_length,polyA,,1,GRC_R,SAMN03112928,,,,,,,"PMID:25775214, We extracted RNA from whole bod...",not exposed to parasite,,ANN,2026-04-21
62199,SRX7400229,SRP237993,Illumina HiSeq 2500,SRS5847895,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,polyA,,,SFM.VNO.F.103,SAMN13618368,,,,,,,PMID: 32770934,,,SAC,2026-04-21
62200,SRX7400228,SRP237993,Illumina HiSeq 2500,SRS5847893,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,polyA,,,SFM.VNO.F.102,SAMN13618367,,,,,,,PMID: 32770934,,,SAC,2026-04-21
62201,SRX7400227,SRP237993,Illumina HiSeq 2500,SRS5847894,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,F,SFM,,10096,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,polyA,,,SFM.VNO.F.101,SAMN13618366,,,,,,,PMID: 32770934,,,SAC,2026-04-21
62202,SRX7400226,SRP237993,NextSeq 500,SRS5847892,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,PAH,,10093,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,polyA,,,PAH.VNO.Pool,SAMN13618365,,,,,,,PMID: 32770934,,,SAC,2026-04-21
62203,SRX7400225,SRP237993,NextSeq 500,SRS5847890,UBERON:0003367,epithelium of vomeronasal organ,UBERON:0000113,post-juvenile adult stage,,Vomeronasal organ - epithelium,Adult,perfect match,not documented,perfect match,mixed,CAR,,10089,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,polyA,,,CAR.VNO.Pool.2,SAMN13618364,,,,,,,PMID: 32770934,,,SAC,2026-04-21


In [36]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1200,SRP190056,Liver transcription of major urinary proteins ...,RNAseq data of liver from males and females in...,SRA,total,Bgee 1K,66,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,,PRJNA530260,31232499,https://onlinelibrary.wiley.com/doi/full/10.11...,10.1111/mec.15155,,
1201,SRP049070,Biomphalaria glabrata Variation,RNA-Seq Illumina reads from snails (Biomphalar...,SRA,partial,Bgee 1K,36,TruSeq RNA Library Prep Kit,full_length,,PRJNA264063,25775214,https://pmc.ncbi.nlm.nih.gov/articles/PMC4361660/,10.1371/journal.pgen.1005067,,[Bgee curator notes: annotation of juvenile sn...
1202,SRP237993,Vomeronasal organ transcriptomes for 5 mouse s...,"Among rodents, olfaction is crucial for a wide...",SRA,partial,Bgee 1K,22,"NEBNext Ultra RNA Library Prep Kit,Poly(A) mRN...",full_length,,PRJNA596328,32770934,https://pmc.ncbi.nlm.nih.gov/articles/PMC7414754/,10.1186/s12862-020-01662-z,,rejected pacbio library


### add annotations to git

In [37]:
! git pull

Already up to date.


In [38]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [39]:
! git add $git_experiment_path $git_library_path

In [40]:
! git commit -m $commit_message_exp

[develop 8e89463] adding annotated bulk experiment SRP237993
 2 files changed, 23 insertions(+)


In [41]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.68 KiB | 2.68 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   590677c..8e89463  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push